In [6]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset,Dataset

from torch.optim import AdamW
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,                     # if you use it elsewhere
    get_linear_schedule_with_warmup,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)

# optional but nice for live progress:
from tqdm.auto import tqdm
import sys, time

In [10]:
df_bt = pd.read_csv("files/welsh_back_translation_high_similarity.csv",
                    usecols=["back_translated_welsh", "cefr_level"])\
         .rename(columns={"back_translated_welsh": "text"})

In [11]:
# Add missing columns 
df_bt["title"] = "Back-translated A1/A2 sample"
df_bt["lang"] = "cy"
df_bt["source_name"] = "back_translation_pipeline"
df_bt["format"] = "text"
df_bt["category"] = "general"
df_bt["license"] = "CC-BY-SA" 

In [12]:
df_bt["cefr_level"].value_counts()

cefr_level
A1    243
A2    127
Name: count, dtype: int64

In [13]:
df_bt_b1b2 = pd.read_csv("files/high_cosine_welsh_back_translation_b1_b2.csv",
                    usecols=["back_translated_welsh", "cefr_level"])\
         .rename(columns={"back_translated_welsh": "text"})

In [14]:
# Add missing columns
df_bt_b1b2["title"] = "Back-translated B1/B2 sample"
df_bt_b1b2["lang"] = "cy"
df_bt_b1b2["source_name"] = "back_translation_pipeline"
df_bt_b1b2["format"] = "text"
df_bt_b1b2["category"] = "general"
df_bt_b1b2["license"] = "CC-BY-SA"

In [15]:
df_bt_b1b2

,text,cefr_level,title,lang,source_name,format,category,license
0,Mae fy mrawd iau yn gweithio eithriad eithriad.,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1,Nid yw fy chwaer yn gweithio ar hyn o bryd.,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
2,Ydw i'n cymdogion gyda chi?,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
3,Sut mae cymdogion yn gallu helpu ei gilydd?,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
4,Rwyt ti wedi bod problem gyda'ch cymydog? Rhy ...,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
...,...,...,...,...,...,...,...,...
376,Mae hi'n dweud ei fod yn heno yn y neuadd heno.,B2,Back-translated B1/B2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
377,Mae'n meddwl ei fod yn ychydig yn gyfrinach.,B2,Back-translated B1/B2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
378,"Dydw i ddim yn gweithio bob dydd, rwy'n mynd i...",B2,Back-translated B1/B2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
379,Dylech gofio'r drws bob amser.,B2,Back-translated B1/B2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA


In [16]:
# Welsh CEFR dataset from HuggingFace
ds = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]
df_main = ds.to_pandas()

In [17]:
df_main["cefr_level"].value_counts()

cefr_level
A1    764
A2    608
Name: count, dtype: int64

In [18]:
# Load your B2 JSON data
df_b1 = pd.read_json("files/B1_canolradd_de-learnwelsh.json")
df_b1["cefr_level"] = "B1"
df_b1 = df_b1.drop_duplicates(subset="text", keep="first")

In [19]:
df_b1

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Adolygu 2,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Mae fy nhad i'n dod o Dreorci.
1,Uned 1 - Adolygu 3,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Mae fy mrawd i'n gweithio men garej.
2,Uned 1 - Adolygu 4,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Dyw fy chwaer i ddim yn gweithio ar hyn o bryd.
3,Uned 1 - Adolygu 5,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Roedd fy nhad-cu i'n gweithio ar fferm.
4,Uned 1 - Siaradwch - Trafod pwnc - Cymdogion 1,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Oes cymdogion da gyda chi?
...,...,...,...,...,...,...,...,...
815,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,"Ar bwy, yn wir, mae’r bai?"
816,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,"Mae iaith yn mynd yn ieithoedd,"
817,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,Mae coeden yn troi’n goed...
818,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,"Yr iaith Gymraeg, dw i’n meddwl,"


In [20]:
df_b1["cefr_level"].value_counts()

cefr_level
B1    802
Name: count, dtype: int64

In [21]:
# Your B2 JSON
df_b2 = pd.read_json("files/b2_welsh.json")
df_b2["cefr_level"] = "B2"

In [22]:
df_b2["cefr_level"].value_counts()

cefr_level
B2    655
Name: count, dtype: int64

In [23]:
# Combine both DataFrames
df_combined = pd.concat([df_main,df_b1,df_b2,df_bt,df_bt_b1b2], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset="text", keep="first")

# Convert back to HuggingFace Dataset
ds_merged = Dataset.from_pandas(df_combined)

In [24]:
df_combined["cefr_level"].value_counts()

cefr_level
B1    982
A1    964
B2    820
A2    719
Name: count, dtype: int64

In [25]:
# 1) Build df_work with 'labels'
CEFR_LEVELS = ["A1","A2","B1","B2"]
label2id = {lab:i for i, lab in enumerate(CEFR_LEVELS)}

df_work = df_combined[df_combined["cefr_level"].isin(CEFR_LEVELS)].copy()
df_work["labels"] = df_work["cefr_level"].map(label2id).astype(int)

# (optional sanity)
print(df_work.columns)  # should include: 'text', 'cefr_level', 'labels'

Index(['title', 'lang', 'source_name', 'format', 'category', 'cefr_level',
       'license', 'text', 'labels'],
      dtype='object')


In [26]:
ds_all = Dataset.from_pandas(
    df_work[["text", "labels"]],
    preserve_index=False
)

In [27]:
ds_all

Dataset({
    features: ['text', 'labels'],
    num_rows: 3485
})

In [28]:
class CEFRDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.labels = df["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [29]:
# Model
class XLMR_MultiProto(nn.Module):
    def __init__(self, model_name, num_labels=4, num_prototypes=5, lm_layer=-2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.num_labels, self.K, self.lm_layer = num_labels, num_prototypes, lm_layer
        d = self.encoder.config.hidden_size
        # random init; we will NOT do any centroid/K-means init
        self.prototypes = nn.Parameter(torch.randn(num_labels, num_prototypes, d))
        # start a bit lower than 10 but not too cold
        self.tau = nn.Parameter(torch.tensor(8.0))

    def mean_pool(self, hidden, mask):
        m = mask.unsqueeze(-1).float()
        return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask, labels=None, class_weights=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           output_hidden_states=(self.lm_layer != -1))
        hidden = out.last_hidden_state if self.lm_layer == -1 else out.hidden_states[self.lm_layer]

        h = F.normalize(self.mean_pool(hidden, attention_mask), dim=-1)   # [B,d]
        P = F.normalize(self.prototypes, dim=-1)                          # [C,K,d]
        sims = torch.einsum("bd,ckd->bck", h, P)                          # [B,C,K]
        logits = sims.mean(dim=2) * self.tau                              # [B,C]

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, weight=class_weights)
        return {"loss": loss, "logits": logits}

In [30]:
def make_optimizer(model, encoder_lr=2e-5, proto_lr=8e-4, tau_lr=1e-3, weight_decay=0.01):
    # No weight decay on tau; optional to exclude on prototypes too, but keep WD on encoder.
    return AdamW(
        [
            {"params": model.encoder.parameters(), "lr": encoder_lr, "weight_decay": weight_decay},
            {"params": [model.prototypes],         "lr": proto_lr,   "weight_decay": weight_decay},
            {"params": [model.tau],                "lr": tau_lr,     "weight_decay": 0.0},
        ]
    )

In [31]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    pr, rc, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )
    metrics = {}
    for i, lab in enumerate(CEFR_LEVELS):
        metrics[f"eval_{lab}_precision"] = pr[i]
        metrics[f"eval_{lab}_recall"]    = rc[i]
        metrics[f"eval_{lab}_f1"]        = f1[i]

    metrics["eval_accuracy"]           = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"]        = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"]    = recall_score(labels, preds, average="weighted")
    return metrics


In [32]:
from transformers import Trainer

class ProtoTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=labels,
            class_weights=self.class_weights
        )
        loss = out["loss"]
        return (loss, out) if return_outputs else loss

In [33]:
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
data_collator = DataCollatorWithPadding(tokenizer)

def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=128)
    enc["labels"] = batch["labels"]  # <- keep labels!
    return enc


In [34]:
def compute_class_weights_from_ds(ds, num_classes):
    labels = np.array(ds["labels"])
    counts = pd.Series(labels).value_counts().to_dict()
    freq = np.array([counts.get(i,0) for i in range(num_classes)], dtype=np.float32)
    freq = np.clip(freq, 1, None)
    gamma = 0.75
    w = (1.0/freq)**gamma
    w = w * (len(w)/w.sum())
    return torch.tensor(w, dtype=torch.float32)

In [235]:
# @torch.no_grad()
# def compute_fold_embeddings(encoder, ds_tok, tokenizer, lm_layer=-1, device="cpu"):
#     encoder.eval()
#     collate = DataCollatorWithPadding(tokenizer)
#     loader = DataLoader(ds_tok, batch_size=64, shuffle=False, collate_fn=collate)  # ← key fix

#     H, Y = [], []
#     for batch in loader:
#         ids  = batch["input_ids"].to(device)
#         mask = batch["attention_mask"].to(device)
#         labs = batch["labels"]  # keep on CPU; we only concat later

#         if lm_layer == -1:
#             out = encoder(input_ids=ids, attention_mask=mask)
#             hidden = out.last_hidden_state
#         else:
#             out = encoder(input_ids=ids, attention_mask=mask, output_hidden_states=True)
#             hidden = out.hidden_states[lm_layer]

#         m = mask.unsqueeze(-1).float()
#         h = (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)   # mean pool
#         h = F.normalize(h, dim=-1)                           # cosine-friendly

#         H.append(h.cpu())
#         Y.append(labs.cpu())

#     return torch.cat(H), torch.cat(Y)


# @torch.no_grad()
# def init_prototypes_from_centroids(model, H, Y, K=3, jitter=0.02, seed=42):
#     g = torch.Generator().manual_seed(seed)
#     d = H.shape[1]; C = model.num_labels
#     P = torch.zeros(C, K, d)
#     for c in range(C):
#         if (Y == c).any():
#             mu = H[Y == c].mean(0)
#         else:
#             mu = torch.randn(d, generator=g)
#             mu = F.normalize(mu, dim=0)
#         for k in range(K):
#             P[c, k] = F.normalize(mu + jitter * torch.randn(d, generator=g), dim=0)
#     model.prototypes.copy_(P)


In [41]:
from transformers import TrainingArguments, EarlyStoppingCallback
from sklearn.model_selection import StratifiedKFold

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

labels_np = np.array(ds_all["labels"])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_rows, best_f1, best_trainer = [], -1.0, None

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels_np)), labels_np), start=1):
    print(f"\nRunning Fold {fold} (train={len(tr_idx)}, val={len(va_idx)})")

    ds_train = ds_all.select(tr_idx)
    ds_val   = ds_all.select(va_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=["text"])
    tok_val   = ds_val.map(preprocess,   batched=True, remove_columns=["text"])

    # model: unfreeze encoder, use penultimate layer, K=5, no init
    model = XLMR_MultiProto(
        MODEL_NAME, num_labels=len(CEFR_LEVELS),
        num_prototypes=5, lm_layer=-2
    ).to(device)
    for p in model.encoder.parameters():
        p.requires_grad = True

    # class weights per fold (or compute once globally and reuse)
    class_w = compute_class_weights_from_ds(ds_train, num_classes=len(CEFR_LEVELS)).to(device)

    # optimizer: differential LRs, no WD on tau
    opt = make_optimizer(model,
                         encoder_lr=2e-5,
                         proto_lr=8e-4,
                         tau_lr=1e-3,
                         weight_decay=0.01)

   # training args: 5 epochs, warmup 0.1, linear decay; mixed precision if CUDA
    args = TrainingArguments(
        output_dir=f"./runs_cefr_mp_b1b2_DA_kfold/fold_{fold}",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,

        eval_strategy="epoch",     # ← correct arg name
        logging_strategy="epoch",
        save_strategy="epoch",           # ← must save checkpoints
        save_total_limit=1,              # keep it tight

        load_best_model_at_end=True,     # ← required for EarlyStopping + best model
        metric_for_best_model="eval_weighted_f1",  # ← must match compute_metrics key
        greater_is_better=True,          # weighted F1 higher is better

        push_to_hub=False,
        report_to=[],                    # no TB/W&B
        learning_rate=2e-5,              # HF still wants a value even if you pass a custom opt
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        gradient_accumulation_steps=1,
        fp16=torch.cuda.is_available(),
        bf16=False,
        seed=42
    )

    trainer = ProtoTrainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=lambda p: compute_metrics((p.predictions, p.label_ids)),
        class_weights=class_w,
        optimizers=(opt, None),  # use our optimizer; Trainer makes the scheduler
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2,
                                        early_stopping_threshold=1e-4)]
    )

    trainer.train()
    metr = trainer.evaluate()

    if metr.get("eval_weighted_f1", 0.0) > best_f1:
        best_f1, best_trainer = metr["eval_weighted_f1"], trainer

    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metr.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall":    metr.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1":        metr.get("eval_weighted_f1", 0.0),
    }
    for lvl in CEFR_LEVELS:
        row[f"{lvl} Precision"] = metr.get(f"eval_{lvl}_precision", 0.0)
        row[f"{lvl} Recall"]    = metr.get(f"eval_{lvl}_recall", 0.0)
        row[f"{lvl} F1"]        = metr.get(f"eval_{lvl}_f1", 0.0)
    all_rows.append(row)

# Summary table
import pandas as pd
df_res = pd.DataFrame(all_rows)
avg = df_res.drop(columns=["Fold"]).mean(numeric_only=True)
avg["Fold"] = "Average"
df_res = pd.concat([df_res, pd.DataFrame([avg])], ignore_index=True)
print(df_res)


Running Fold 1 (train=2788, val=697)


Map: 100%|██████████| 697/697 [00:00<00:00, 17613.57 examples/s]
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
C:\Users\eesha\AppData\Local\Temp\ipykernel_21544\2513505774.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `ProtoTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,Accuracy,Weighted F1,Weighted Precision,Weighted Recall
1,1.283600,1.074915,0.527273,0.751295,0.619658,0.293333,0.152778,0.200913,0.859375,0.561224,0.679012,0.534247,0.713415,0.610966,0.565280,0.547791,0.573971,0.565280
2,0.955700,0.893910,0.565371,0.829016,0.672269,0.589286,0.229167,0.330000,0.890411,0.663265,0.760234,0.584906,0.756098,0.659574,0.641320,0.623306,0.666311,0.641320
3,0.805800,0.851266,0.654709,0.756477,0.701923,0.491935,0.423611,0.455224,0.908451,0.658163,0.763314,0.591346,0.750000,0.661290,0.658537,0.658658,0.677524,0.658537


c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Running Fold 2 (train=2788, val=697)


...


Running Fold 5 (train=2788, val=697)


Map: 100%|██████████| 697/697 [00:00<00:00, 25640.97 examples/s]
C:\Users\eesha\AppData\Local\Temp\ipykernel_21544\2513505774.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `ProtoTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,Accuracy,Weighted F1,Weighted Precision,Weighted Recall
1,1.275600,1.048777,0.495652,0.890625,0.636872,0.517857,0.201389,0.290000,0.827338,0.583756,0.684524,0.662420,0.634146,0.647975,0.601148,0.581289,0.633227,0.601148
2,0.940900,0.841366,0.605166,0.854167,0.708423,0.591398,0.381944,0.464135,0.863354,0.705584,0.776536,0.697674,0.731707,0.714286,0.685796,0.678584,0.697062,0.685796
3,0.778900,0.796275,0.672727,0.770833,0.718447,0.572464,0.548611,0.560284,0.892405,0.715736,0.794366,0.685083,0.756098,0.718841,0.705882,0.707321,0.717010,0.705882


c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


      Fold  All CEFR Levels Precision  All CEFR Levels Recall  \
0        1                   0.677524                0.658537   
1        2                   0.707580                0.692970   
2        3                   0.717100                0.701578   
3        4                   0.751140                0.736011   
4        5                   0.717010                0.705882   
5  Average                   0.714071                0.698996   

   All CEFR Levels F1  A1 Precision  A1 Recall     A1 F1  A2 Precision  \
0            0.658658      0.654709   0.756477  0.701923      0.491935   
1            0.688902      0.633858   0.834197  0.720358      0.537736   
2            0.702576      0.642857   0.839378  0.728090      0.555556   
3            0.737924      0.716157   0.849741  0.777251      0.596154   
4            0.707321      0.672727   0.770833  0.718447      0.572464   
5            0.699076      0.664062   0.810125  0.729214      0.550769   

   A2 Recall     A2 F1  B

In [42]:
# Convert to DataFrame
df = pd.DataFrame(df_res)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
]

df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)

In [43]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.677524  0.658537  0.658658  0.654709  0.756477  0.701923   
1        2        0.707580  0.692970  0.688902  0.633858  0.834197  0.720358   
2        3        0.717100  0.701578  0.702576  0.642857  0.839378  0.728090   
3        4        0.751140  0.736011  0.737924  0.716157  0.849741  0.777251   
4        5        0.717010  0.705882  0.707321  0.672727  0.770833  0.718447   
5  Average        0.714071  0.698996  0.699076  0.664062  0.810125  0.729214   

         A2                            B1                            B2  \
  Precision    Recall        F1 Precision    Recall        F1 Precision   
0  0.491935  0.423611  0.455224  0.908451  0.658163  0.763314  0.591346   
1  0.537736  0.395833  0.456000  0.939189  0.709184  0.808140  0.666667   
2  0.555556  0.555556  0.555556  0.888889  0.734694  0.804469  0.741007   
3  0.596154  0.650350  0.622074  0.914474  0.705584  0.796562  0.731250   
4  0.572464  0.548611  0.560284  0.892405  0.715736  0.794366  0.685083   
5  0.550769  0.514792  0.529827  0.908682  0.704672  0.793370  0.683071   

                       
     Recall        F1  
0  0.750000  0.661290  
1  0.768293  0.713881  
2  0.628049  0.679868  
3  0.713415  0.722222  
4  0.756098  0.718841  
5  0.723171  0.699220

In [37]:
import os, glob, json
import numpy as np
import torch
from safetensors.torch import load_file as safe_load
from transformers import TrainingArguments
from sklearn.metrics import confusion_matrix


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def latest_ckpt_dir(fold_dir: str) -> str:
    """Pick the checkpoint-* dir with the largest step number."""
    cand = glob.glob(os.path.join(fold_dir, "checkpoint-*"))
    if not cand:
        raise FileNotFoundError(f"No checkpoint-* found in {fold_dir}")
    # sort by integer step
    cand.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
    return cand[-1]

def load_weights_into_model(model, ckpt_dir: str):
    safepath = os.path.join(ckpt_dir, "model.safetensors")
    ptpath   = os.path.join(ckpt_dir, "pytorch_model.bin")
    if os.path.exists(safepath):
        state = safe_load(safepath, device="cpu")
    elif os.path.exists(ptpath):
        state = torch.load(ptpath, map_location="cpu")
    else:
        raise FileNotFoundError(f"No model.safetensors or pytorch_model.bin in {ckpt_dir}")

    # Handle possible 'module.' prefix from DP/DDP
    def strip_module_prefix(sd):
        if not sd: return sd
        if all(k.startswith("module.") for k in sd.keys()):
            return {k.replace("module.", "", 1): v for k,v in sd.items()}
        return sd

    state = strip_module_prefix(state)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f"[warn] load_state_dict: missing={len(missing)}, unexpected={len(unexpected)}")
        if missing:   print("  missing keys (first 10):", missing[:10])
        if unexpected:print("  unexpected keys (first 10):", unexpected[:10])

    model.to(device).eval()
    return model

all_cms = []
labels_order = list(range(len(CEFR_LEVELS)))


labels_np = np.array(ds_all["labels"])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels_np)), labels_np), start=1):
    fold_root = f"./runs_cefr_mp_b1b2_DA_kfold/fold_{fold}"
    ckpt_dir  = latest_ckpt_dir(fold_root)
    print(f"\nFold {fold}: using checkpoint → {ckpt_dir}")

    # rebuild validation set for this fold
    ds_val  = ds_all.select(va_idx)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=["text"])

    # rebuild the model with SAME init args as training
    model = XLMR_MultiProto(
        MODEL_NAME, num_labels=len(CEFR_LEVELS),
        num_prototypes=5, lm_layer=-2
    )
    model = load_weights_into_model(model, ckpt_dir)

    # minimal Trainer for inference
    args = TrainingArguments(
        output_dir=fold_root,
        per_device_eval_batch_size=32,
        do_train=False, do_eval=True,
        report_to=[]
    )

    trainer = ProtoTrainer(
        model=model,
        args=args,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=None,
    )

    # predict on this fold's val split
    pred_out = trainer.predict(tok_val)
    y_true = pred_out.label_ids
    if pred_out.predictions.ndim > 1:
        y_pred = pred_out.predictions.argmax(axis=1)
    else:
        y_pred = (pred_out.predictions > 0.5).astype(int)

    # confusion matrix for this fold
    cm = confusion_matrix(y_true, y_pred, labels=labels_order)
    np.savetxt(os.path.join(ckpt_dir, "confusion_matrix.csv"), cm, fmt="%d", delimiter=",")
    print(f"Fold {fold} confusion matrix:\n{cm}")
    all_cms.append(cm)

# ---- Overall (micro-aggregated) confusion matrix ----
overall_cm = sum(all_cms)
np.savetxt("./runs_cefr_mp_kfold/confusion_matrix_overall.csv", overall_cm, fmt="%d", delimiter=",")
print("\nOverall confusion matrix (summed across folds):\n", overall_cm)

# ---- Optional: also save a row-normalized version for readability ----
row_sums = overall_cm.sum(axis=1, keepdims=True).clip(min=1)
overall_cm_norm = overall_cm / row_sums
np.savetxt("./runs_cefr_mp_kfold/confusion_matrix_overall_normalized.csv", overall_cm_norm, fmt="%.6f", delimiter=",")
print("Row-normalized overall CM (each row sums to 1):\n", np.round(overall_cm_norm, 3))



Fold 1: using checkpoint → ./runs_cefr_mp_b1b2_DA_kfold/fold_1\checkpoint-525


Map: 100%|██████████| 697/697 [00:00<00:00, 12908.57 examples/s]
C:\Users\eesha\AppData\Local\Temp\ipykernel_17636\2513505774.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `ProtoTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Fold 1 confusion matrix:
[[146  34   3  10]
 [ 42  61   6  35]
 [ 17  10 129  40]
 [ 18  19   4 123]]

Fold 2: using checkpoint → ./runs_cefr_mp_b1b2_DA_kfold/fold_2\checkpoint-525


Map: 100%|██████████| 697/697 [00:00<00:00, 16096.14 examples/s]
C:\Users\eesha\AppData\Local\Temp\ipykernel_17636\2513505774.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `ProtoTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Fold 2 confusion matrix:
[[161  15   4  13]
 [ 56  57   1  30]
 [ 22  15 139  20]
 [ 15  19   4 126]]

Fold 3: using checkpoint → ./runs_cefr_mp_b1b2_DA_kfold/fold_3\checkpoint-525


...

Fold 3 confusion matrix:
[[162  22   4   5]
 [ 41  80   4  19]
 [ 28  12 144  12]
 [ 21  30  10 103]]

Fold 4: using checkpoint → ./runs_cefr_mp_b1b2_DA_kfold/fold_4\checkpoint-525


Map: 100%|██████████| 697/697 [00:00<00:00, 19582.75 examples/s]
C:\Users\eesha\AppData\Local\Temp\ipykernel_17636\2513505774.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `ProtoTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Fold 4 confusion matrix:
[[164  21   2   6]
 [ 32  93   3  15]
 [ 17  19 139  22]
 [ 16  23   8 117]]

Fold 5: using checkpoint → ./runs_cefr_mp_b1b2_DA_kfold/fold_5\checkpoint-525


Map: 100%|██████████| 697/697 [00:00<00:00, 20330.12 examples/s]
C:\Users\eesha\AppData\Local\Temp\ipykernel_17636\2513505774.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `ProtoTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
c:\Users\eesha\Desktop\RA(UniversalCEFR)\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Fold 5 confusion matrix:
[[148  32   9   3]
 [ 37  79   3  25]
 [ 16  11 141  29]
 [ 19  16   5 124]]

Overall confusion matrix (summed across folds):
 [[781 124  22  37]
 [208 370  17 124]
 [100  67 692 123]
 [ 89 107  31 593]]
Row-normalized overall CM (each row sums to 1):
 [[0.81  0.129 0.023 0.038]
 [0.289 0.515 0.024 0.172]
 [0.102 0.068 0.705 0.125]
 [0.109 0.13  0.038 0.723]]


In [38]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report


# set your labels in the same order as label_ids
labels = CEFR_LEVELS  # e.g. ["A1","A2","B1","B2"]

def plot_cm(cm, labels, title, outfile, normalize=None):
    cm_to_plot = cm.astype(float)
    if normalize == 'true':      # row-normalize
        cm_to_plot = cm_to_plot / cm_to_plot.sum(axis=1, keepdims=True).clip(min=1)
        fmt = ".2f"
    elif normalize == 'pred':    # col-normalize
        cm_to_plot = cm_to_plot / cm_to_plot.sum(axis=0, keepdims=True).clip(min=1)
        fmt = ".2f"
    else:
        fmt = "d"   # counts (integers)

    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(cm_to_plot, aspect='auto')
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels, yticklabels=labels,
        ylabel="Actual", xlabel="Predicted",
        title=title
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # annotate
    thresh = cm_to_plot.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = cm_to_plot[i, j]
            if fmt == "d":
                text = f"{int(val):d}"
            else:
                text = f"{val:{fmt}}"
            ax.text(j, i, text,
                    ha="center", va="center",
                    color="white" if val > thresh else "black")

    fig.tight_layout()
    fig.savefig(outfile, dpi=220)
    plt.close(fig)

def save_classification_report(y_true, y_pred, labels, outfile_csv):
    # Map ids→names for readability in the CSV
    target_names = labels
    rep_dict = classification_report(
        y_true, y_pred, labels=list(range(len(labels))),
        target_names=target_names, output_dict=True, zero_division=0
    )
    # write to CSV (simple)
    import csv
    with open(outfile_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        # header
        keys = list(next(iter(rep_dict.values())).keys()) if isinstance(next(iter(rep_dict.values())), dict) else []
        # flatten dict
        w.writerow(["class/avg"] + (keys if keys else ["precision","recall","f1-score","support"]))
        for k, v in rep_dict.items():
            if isinstance(v, dict):
                w.writerow([k] + [v.get("precision",""), v.get("recall",""), v.get("f1-score",""), v.get("support","")])
            else:
                # fallback (rare)
                w.writerow([k, v])


plot_cm(cm, labels, f"Fold {fold} – Confusion Matrix (Counts)",
        os.path.join(ckpt_dir, "cm_counts.png"), normalize=None)

plot_cm(cm, labels, f"Fold {fold} – Confusion Matrix (Row-Normalized)",
        os.path.join(ckpt_dir, "cm_row_norm.png"), normalize='true')

# Also save a precision-oriented view (column-normalized), optional:
plot_cm(cm, labels, f"Fold {fold} – Confusion Matrix (Col-Normalized)",
        os.path.join(ckpt_dir, "cm_col_norm.png"), normalize='pred')

# Per-fold classification report
save_classification_report(y_true, y_pred, labels,
        os.path.join(ckpt_dir, "classification_report.csv"))

# But if you only have CMs, still plot aggregated:
plot_cm(overall_cm, labels, "Overall Confusion Matrix (Counts)",
        "./runs_cefr_mp_kfold/confusion_matrix_overall_counts.png", normalize=None)

plot_cm(overall_cm, labels, "Overall Confusion Matrix (Row-Normalized)",
        "./runs_cefr_mp_kfold/confusion_matrix_overall_row_norm.png", normalize='true')